In [1]:
import copernicusmarine as cm
import xarray as xr
from datetime import datetime

In [2]:
variables = ('o2', 'nppv', 'chl')

In [3]:
from typing import Union, Iterable

In [4]:
CatalogueTree = Union[dict[str, Iterable['CatalogueTree']], list[Iterable['CatalogueTree']], str, list[float]]

In [5]:
catalogue = cm.describe(include_datasets=True, contains=["med-ogs-bio-rean-d"])

Fetching catalog: 100%|██████████| 3/3 [00:11<00:00,  3.93s/it]


In [6]:
catalogue

{'products': [{'title': 'Mediterranean Sea Biogechemistry Reanalysis',
   'product_id': 'MEDSEA_MULTIYEAR_BGC_006_008',
   'thumbnail_url': 'https://catalogue.marine.copernicus.eu/documents/IMG/MEDSEA_MULTIYEAR_BGC_006_008.png',
   'digital_object_identifier': '10.25423/cmcc/medsea_multiyear_bgc_006_008_medbfm3',
   'sources': ['Numerical models'],
   'processing_level': 'Level 4',
   'production_center': 'OGS (Italy)',
   'datasets': [{'dataset_id': 'med-ogs-bio-rean-d',
     'dataset_name': 'Primary Production and Oxygen (3D) - Daily Mean',
     'versions': [{'label': '202105',
       'parts': [{'name': 'default',
         'services': [{'service_type': {'service_name': 'original-files',
            'short_name': 'files'},
           'service_format': None,
           'uri': 'https://s3.waw3-1.cloudferro.com/mdl-native-12/native/MEDSEA_MULTIYEAR_BGC_006_008/med-ogs-bio-rean-d_202105',
           'variables': [{'short_name': 'nppv',
             'standard_name': 'net_primary_production

In [10]:
from jax.tree_util import tree_structure

In [11]:
tree_structure(catalogue)

PyTreeDef({'products': [{'datasets': [{'dataset_id': *, 'dataset_name': *, 'versions': [{'label': *, 'parts': [{'name': *, 'released_date': None, 'retired_date': None, 'services': [{'service_format': None, 'service_type': {'service_name': *, 'short_name': *}, 'uri': *, 'variables': [{'bbox': [*, *, *, *], 'coordinates': [], 'short_name': *, 'standard_name': *, 'units': *}, {'bbox': [*, *, *, *], 'coordinates': [], 'short_name': *, 'standard_name': *, 'units': *}]}, {'service_format': *, 'service_type': {'service_name': *, 'short_name': *}, 'uri': *, 'variables': [{'bbox': [*, *, *, *], 'coordinates': [{'chunk_geometric_factor': None, 'chunk_reference_coordinate': None, 'chunk_type': None, 'chunking_length': *, 'coordinates_id': *, 'maximum_value': None, 'minimum_value': None, 'step': None, 'units': *, 'values': [*, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *, *,

In [9]:
cm.open_dataset(dataset_id=f"med-ogs-bio-rean-d", variables=("nppv", "o2"))

INFO - 2024-05-20T15:14:05Z - Dataset version was not specified, the latest one was selected: "202105"
INFO - 2024-05-20T15:14:05Z - Dataset part was not specified, the first one was selected: "default"
INFO - 2024-05-20T15:14:07Z - Service was not specified, the default one was selected: "arco-geo-series"


<xarray.Dataset> Size: 3TB
Dimensions:    (depth: 125, latitude: 380, longitude: 1005, time: 8613)
Coordinates:
  * depth      (depth) float32 500B 1.018 3.166 5.465 ... 4.062e+03 4.153e+03
  * latitude   (latitude) float32 2kB 30.19 30.23 30.27 ... 45.9 45.94 45.98
  * longitude  (longitude) float32 4kB -5.542 -5.5 -5.458 ... 36.21 36.25 36.29
  * time       (time) datetime64[ns] 69kB 1999-01-01 1999-01-02 ... 2022-07-31
Data variables:
    nppv       (time, depth, latitude, longitude) float32 2TB ...
    o2         (time, depth, latitude, longitude) float32 2TB ...
Attributes:
    Conventions:    CF-1.0
    bulletin_date:  2021-03-23
    bulletin_type:  analysis
    comment:        Please check in CMEMS catalogue the INFO section for prod...
    contact:        servicedesk.cmems@mercator-ocean.eu
    field_type:     daily_mean_centered_at_time_field
    institution:    OGS (Istituto Nazionale di Oceanografia e di Geofisica Sp...
    references:     Please check in CMEMS catalogue the INFO section for prod...
    source:         3DVAR-OGSTM-BFM
    title:          Primary Production and Oxygen (3D) - Daily Mean

In [11]:
def find_matching_subtree(tree: CatalogueTree, search_key: str, valid_values: list[str]) -> CatalogueTree:
    matches = []
    _find_matching_subtrees(matches, tree, search_key, valid_values)
    return matches


def _find_matching_subtrees(matches: CatalogueTree, tree: CatalogueTree, search_key: str, valid_values: list[str]) -> CatalogueTree:
    if isinstance(tree, list):
        for subtree in tree:
            _find_matching_subtrees(matches, subtree, search_key, valid_values)
    elif isinstance(tree, dict):
        if search_key in tree.keys() and tree[search_key] in valid_values:
            matches.append(tree)
        else:
            for subtree in tree.values():
                _find_matching_subtrees(matches, subtree, search_key, valid_values)


def find_variables_in_dataset(catalogue, dataset_id, variables):
    variables_in_dataset = []
    dataset = find_matching_subtree(catalogue, 'dataset_id', [dataset_id])
    for variable in variables:
        if find_matching_subtree(dataset, 'short_name', [variable]):
            variables_in_dataset.append(variable)
    return variables_in_dataset

In [12]:
find_variables_in_dataset(catalogue, "med-ogs-bio-rean-d", variables)

['o2', 'nppv']

In [2]:
start_datetime = datetime.fromisoformat('2000-01-01')
end_datetime = datetime.fromisoformat('2000-01-02')

latitude_start = -5.541667
latitude_end = 36.2917

longitude_start = 30.1875
longitude_end = 45.97917

datasets = {'bio': ['o2', 'nppv']}

In [3]:
def download_rean(dataset_id_code, variables):
    return cm.open_dataset(dataset_id=f"med-{dataset_id_code}-rean-d", start_datetime=start_datetime, end_datetime=end_datetime, variables=variables)

In [4]:
ds_ogs = download_rean('ogs-bio', ['o2', 'nppv'])

Fetching catalog: 100%|█████████████████████████████████████████████████████████████████| 3/3 [00:10<00:00,  3.53s/it]


INFO - 2024-05-10T14:11:21Z - Dataset version was not specified, the latest one was selected: "202105"
INFO - 2024-05-10T14:11:21Z - Dataset part was not specified, the first one was selected: "default"
INFO - 2024-05-10T14:11:24Z - Service was not specified, the default one was selected: "arco-geo-series"


In [5]:
ds_ogs

<xarray.Dataset> Size: 764MB
Dimensions:    (depth: 125, latitude: 380, longitude: 1005, time: 2)
Coordinates:
  * depth      (depth) float32 500B 1.018 3.166 5.465 ... 4.062e+03 4.153e+03
  * latitude   (latitude) float32 2kB 30.19 30.23 30.27 ... 45.9 45.94 45.98
  * longitude  (longitude) float32 4kB -5.542 -5.5 -5.458 ... 36.21 36.25 36.29
  * time       (time) datetime64[ns] 16B 2000-01-01 2000-01-02
Data variables:
    o2         (time, depth, latitude, longitude) float32 382MB ...
    nppv       (time, depth, latitude, longitude) float32 382MB ...
Attributes:
    Conventions:    CF-1.0
    bulletin_date:  2021-03-23
    bulletin_type:  analysis
    comment:        Please check in CMEMS catalogue the INFO section for prod...
    contact:        servicedesk.cmems@mercator-ocean.eu
    field_type:     daily_mean_centered_at_time_field
    institution:    OGS (Istituto Nazionale di Oceanografia e di Geofisica Sp...
    references:     Please check in CMEMS catalogue the INFO section for prod...
    source:         3DVAR-OGSTM-BFM
    title:          Primary Production and Oxygen (3D) - Daily Mean

In [6]:
ds_cmcc = download_rean('cmcc-cur', ['uo', 'vo'])

INFO - 2024-05-10T14:11:37Z - Dataset version was not specified, the latest one was selected: "202012"
INFO - 2024-05-10T14:11:37Z - Dataset part was not specified, the first one was selected: "default"
INFO - 2024-05-10T14:11:39Z - Service was not specified, the default one was selected: "arco-geo-series"


In [7]:
ds_cmcc

<xarray.Dataset> Size: 871MB
Dimensions:    (depth: 141, latitude: 380, longitude: 1016, time: 2)
Coordinates:
  * depth      (depth) float32 564B 1.018 3.166 5.465 ... 5.646e+03 5.754e+03
  * latitude   (latitude) float32 2kB 30.19 30.23 30.27 ... 45.9 45.94 45.98
  * longitude  (longitude) float32 4kB -6.0 -5.958 -5.917 ... 36.21 36.25 36.29
  * time       (time) datetime64[ns] 16B 2000-01-01 2000-01-02
Data variables:
    uo         (time, depth, latitude, longitude) float32 435MB ...
    vo         (time, depth, latitude, longitude) float32 435MB ...
Attributes:
    Conventions:    CF-1.0
    bulletin_date:  20200901
    bulletin_type:  reanalysis
    comment:        Please check in CMEMS catalogue the INFO section for prod...
    contact:        servicedesk.cmems@mercator-ocean.eu
    field_type:     daily_mean_centered_at_time_field
    institution:    Centro Euro-Mediterraneo sui Cambiamenti Climatici - CMCC...
    references:     Please check in CMEMS catalogue the INFO section for prod...
    source:         MFS E3R1
    title:          Horizontal Velocity (3D) - Daily Mean

In [21]:
ds_cmcc.sel(depth=ds_ogs.coords['depth'])

<xarray.Dataset> Size: 772MB
Dimensions:    (depth: 125, latitude: 380, longitude: 1016, time: 2)
Coordinates:
  * depth      (depth) float32 500B 1.018 3.166 5.465 ... 4.062e+03 4.153e+03
  * latitude   (latitude) float32 2kB 30.19 30.23 30.27 ... 45.9 45.94 45.98
  * longitude  (longitude) float32 4kB -6.0 -5.958 -5.917 ... 36.21 36.25 36.29
  * time       (time) datetime64[ns] 16B 2000-01-01 2000-01-02
Data variables:
    uo         (time, depth, latitude, longitude) float32 386MB ...
    vo         (time, depth, latitude, longitude) float32 386MB ...
Attributes:
    Conventions:    CF-1.0
    bulletin_date:  20200901
    bulletin_type:  reanalysis
    comment:        Please check in CMEMS catalogue the INFO section for prod...
    contact:        servicedesk.cmems@mercator-ocean.eu
    field_type:     daily_mean_centered_at_time_field
    institution:    Centro Euro-Mediterraneo sui Cambiamenti Climatici - CMCC...
    references:     Please check in CMEMS catalogue the INFO section for prod...
    source:         MFS E3R1
    title:          Horizontal Velocity (3D) - Daily Mean

In [33]:
xr.merge([ds_cmcc, ds_ogs])

<xarray.Dataset> Size: 2GB
Dimensions:    (depth: 141, latitude: 380, longitude: 1017, time: 2)
Coordinates:
  * depth      (depth) float32 564B 1.018 3.166 5.465 ... 5.646e+03 5.754e+03
  * latitude   (latitude) float32 2kB 30.19 30.23 30.27 ... 45.9 45.94 45.98
  * longitude  (longitude) float32 4kB -6.0 -5.958 -5.917 ... 36.21 36.25 36.29
  * time       (time) datetime64[ns] 16B 2000-01-01 2000-01-02
Data variables:
    uo         (time, depth, latitude, longitude) float32 436MB nan nan ... nan
    vo         (time, depth, latitude, longitude) float32 436MB nan nan ... nan
    o2         (time, depth, latitude, longitude) float32 436MB nan nan ... nan
    nppv       (time, depth, latitude, longitude) float32 436MB nan nan ... nan
Attributes:
    Conventions:    CF-1.0
    bulletin_date:  20200901
    bulletin_type:  reanalysis
    comment:        Please check in CMEMS catalogue the INFO section for prod...
    contact:        servicedesk.cmems@mercator-ocean.eu
    field_type:     daily_mean_centered_at_time_field
    institution:    Centro Euro-Mediterraneo sui Cambiamenti Climatici - CMCC...
    references:     Please check in CMEMS catalogue the INFO section for prod...
    source:         MFS E3R1
    title:          Horizontal Velocity (3D) - Daily Mean

In [12]:
ds = xr.open_dataset("/Users/stefanocampanella/Downloads/dataset_source-era5_date-2022-01-01_res-0.25_levels-13_steps-01 (1).nc")

In [13]:
ds

<xarray.Dataset> Size: 1GB
Dimensions:                       (lon: 1440, lat: 721, level: 13, time: 3,
                                   batch: 1)
Coordinates:
  * lon                           (lon) float32 6kB 0.0 0.25 0.5 ... 359.5 359.8
  * lat                           (lat) float32 3kB -90.0 -89.75 ... 89.75 90.0
  * level                         (level) int32 52B 50 100 150 ... 850 925 1000
  * time                          (time) timedelta64[ns] 24B 00:00:00 ... 12:...
    datetime                      (batch, time) datetime64[ns] 24B ...
Dimensions without coordinates: batch
Data variables: (12/14)
    geopotential_at_surface       (lat, lon) float32 4MB ...
    land_sea_mask                 (lat, lon) float32 4MB ...
    2m_temperature                (batch, time, lat, lon) float32 12MB ...
    mean_sea_level_pressure       (batch, time, lat, lon) float32 12MB ...
    10m_v_component_of_wind       (batch, time, lat, lon) float32 12MB ...
    10m_u_component_of_wind       (batch, time, lat, lon) float32 12MB ...
    ...                            ...
    temperature                   (batch, time, level, lat, lon) float32 162MB ...
    geopotential                  (batch, time, level, lat, lon) float32 162MB ...
    u_component_of_wind           (batch, time, level, lat, lon) float32 162MB ...
    v_component_of_wind           (batch, time, level, lat, lon) float32 162MB ...
    vertical_velocity             (batch, time, level, lat, lon) float32 162MB ...
    specific_humidity             (batch, time, level, lat, lon) float32 162MB ...

In [14]:
del ds